# Exercise Sheet 0 — Refresher: Probability & Statistics
**Course:** Big Data Science in Finance (WI3436TU)
**Assets:** SPY (S&P 500 ETF), AAPL (Apple Inc.)
**Sample window:** 2022-01-01 to 2025-12-31

This notebook works through Problems 1-4 of Exercise Sheet 0. Code cells download and
process the data and produce the required tables/plots; markdown cells contain the
written derivations and discussion.


## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import yfinance as yf

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

TICKERS = ["SPY", "AAPL"]
START_DATE = "2022-01-01"
END_DATE = "2025-12-31"


In [2]:
def download_prices(ticker, start_date, end_date):
    '''Download raw OHLC(V) + Adj Close data for a single ticker.

    yfinance treats `end` as exclusive, so we push it one day forward to make
    end_date inclusive. If end_date itself is not a trading day, the download
    naturally stops at the closest previous trading day.
    '''
    end_inclusive = (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = yf.download(
        ticker,
        start=start_date,
        end=end_inclusive,
        auto_adjust=False,   # keep Close and Adj Close as separate columns
        progress=False,
    )
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index)
    return df.sort_index()


def choose_price_series(df):
    '''S_t := Adj Close if available, otherwise Close (never mixed within an asset).'''
    if "Adj Close" in df.columns and df["Adj Close"].notna().any():
        col = "Adj Close"
    else:
        col = "Close"
    return df[col].rename("Price"), col


## Problem 1

### a) Summary table

Create a summary table containing, for each asset:
- ticker,
- start date, 
- end date,
- number of downloaded trading days,
- number of missing values in the chosen price column,
- the chosen price column (Adj Close or Close),
- the number n = number of daily log-return observations.

### b) Daily log-returns

Compute the daily log-returns $r_t = log(S_t) − log(S_t−1)$ for each asset. Produce one time series plot for each asset, with trading days on the horizontal axis and daily log-returns on the vertical axis.

### c) Why do we generally prefer returns over prices for statistical analysis?

answer


### d) For one asset, plot both the _Close_ and _Adj Close_ price series on the same figure over the full sample period, using the same vertical axis. Explain what the differences represent and why _Adj Close_ is usually preferred for computing returns.

### e) Additivity of log-returns

**Claim.** For trading days $a<b<c$, deifne $r_{a,b}:=\log S_b-\log S_a$. SHow that: 
$$r_{a,c}=r_{a,b}+r_{b,c}$$

and explain why this additivity is useful when aggregating returns over several trading days.



## Problem 2

### a) Sample statistics

### b) Mean vs median, std vs IQR

**Discussion (read together with the numbers printed above).**

- **Mean vs. median.** If the mean and median are close, the distribution of returns is
  roughly symmetric; a mean noticeably below (above) the median, together with negative
  (positive) sample skewness, indicates left (right) skew — typically a few very negative
  (positive) days pulling the mean away from the more robust median.
- **Std vs. IQR.** For a Gaussian, $\text{std}/\text{IQR}\approx 0.741$. A ratio
  *larger* than this benchmark means the standard deviation is inflated relative to the
  bulk of the data captured by the IQR — i.e. a small number of extreme observations
  (outliers) are driving the dispersion measure that uses all the data (std), while the
  quantile-based IQR is robust to them. This, together with positive excess kurtosis,
  points to **heavy tails**: daily equity returns have far more extreme days than a Normal
  model would predict, so outliers should be expected rather than treated as data errors.


### c) Standardized returns and tail proportions

**Comparison.** The theoretical Gaussian probabilities $\mathbb P(|Z|>3)\approx0.27\%$
and $\mathbb P(|Z|>4)\approx0.0063\%$ are extremely small. The empirical proportions
$\hat p_3,\hat p_4$ computed above are typically several times larger than their Gaussian
counterparts (see the `ratio_p3`/`ratio_p4` columns) — daily stock/ETF returns have
**fatter tails** than the Normal distribution, consistent with the positive excess
kurtosis found in part (a): extreme moves of 3-4 standard deviations happen far more often
in practice than a Gaussian model would predict.


## Problem 3

### a) Empirical quantiles

**Interpretation.** The mean and standard deviation only summarize the *central
tendency* and *average spread* of the return distribution — they say nothing about how bad
the worst days actually are. The 1% and 99% quantiles are the return levels that are
exceeded (on the downside/upside respectively) only 1% of the time; $q^{\text{emp}}_{0.01}$
is a direct empirical estimate of a one-day 99% Value-at-Risk, and $q^{\text{emp}}_{0.99}$
gives the analogous upside magnitude. Because returns are typically skewed and heavy-tailed
(Problem 2), these tail quantiles can be considerably larger in magnitude than what a
Normal approximation using only $\bar r$ and $s$ would imply, and asymmetric between the
lower and upper tail — information that mean and standard deviation alone cannot reveal.


### b) Standardization: mean zero, variance one

**Claim.** Let $R$ be a random variable with $\mathbb E[R]=\mu$, $\operatorname{Var}(R)=\sigma^2<\infty$,
$\sigma>0$, and define $Z=\dfrac{R-\mu}{\sigma}$. Then $\mathbb E[Z]=0$ and $\operatorname{Var}(Z)=1$.

**Proof.**

*Mean.* By linearity of expectation,
$$
\mathbb E[Z]=\mathbb E\!\left[\frac{R-\mu}{\sigma}\right]=\frac{1}{\sigma}\big(\mathbb E[R]-\mu\big)=\frac{1}{\sigma}(\mu-\mu)=0.
$$

*Variance.* For any constants $c\neq 0$ and $d$, $\operatorname{Var}(cR+d)=c^2\operatorname{Var}(R)$
(adding a constant $d$ does not change variance, and scaling by $c$ scales the variance by $c^2$). Here
$Z=\frac{1}{\sigma}R-\frac{\mu}{\sigma}$, so $c=1/\sigma$ and $d=-\mu/\sigma$, giving
$$
\operatorname{Var}(Z)=\left(\frac{1}{\sigma}\right)^{2}\operatorname{Var}(R)=\frac{\sigma^2}{\sigma^2}=1. \qquad\blacksquare
$$


## Problem 4

In [ ]:
ret_df = pd.concat({t: returns[t] for t in TICKERS}, axis=1).dropna()
x = ret_df["SPY"].to_numpy()   # x_t = r_t^SPY
y = ret_df["AAPL"].to_numpy()  # y_t = r_t^AAPL
m = len(x)
print(f"m = {m} trading days with both returns available")


### a) Scatterplot with fitted line

In [ ]:
x_bar, y_bar = x.mean(), y.mean()
beta_hat = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
alpha_hat = y_bar - beta_hat * x_bar

x_grid = np.linspace(x.min(), x.max(), 100)
y_fit_grid = alpha_hat + beta_hat * x_grid

plt.figure(figsize=(6, 6))
plt.scatter(x, y, s=10, alpha=0.5, label="observations")
plt.plot(x_grid, y_fit_grid, color="red", label=f"fit: $y={alpha_hat:.4f}+{beta_hat:.4f}x$")
plt.xlabel(r"SPY log-return $x_t$")
plt.ylabel(r"AAPL log-return $y_t$")
plt.title("AAPL vs SPY daily log-returns")
plt.legend()
plt.tight_layout()
plt.show()


### b) Deriving and verifying the OLS estimators

**Setup.** $(\hat\alpha,\hat\beta)=\displaystyle\operatorname*{arg\,min}_{a,b\in\mathbb R}\sum_{t=1}^{m}(y_t-a-bx_t)^2=:\operatorname*{arg\,min}_{a,b} Q(a,b)$.

**First-order condition in $a$:**
$$
\frac{\partial Q}{\partial a}=-2\sum_{t=1}^m (y_t-a-bx_t)=0
\;\Longrightarrow\;
\sum_t y_t - ma - b\sum_t x_t = 0
\;\Longrightarrow\;
\hat\alpha=\bar y-\hat\beta\bar x. \tag{1}
$$

**First-order condition in $b$:**
$$
\frac{\partial Q}{\partial b}=-2\sum_{t=1}^m x_t(y_t-a-bx_t)=0
\;\Longrightarrow\;
\sum_t x_ty_t - a\sum_t x_t - b\sum_t x_t^2 = 0. \tag{2}
$$

Substituting (1) into (2) and using $\sum_t x_t=m\bar x$:
$$
\sum_t x_ty_t - (\bar y-\hat\beta\bar x)\,m\bar x - \hat\beta\sum_t x_t^2=0
\;\Longrightarrow\;
\hat\beta\Big(\sum_t x_t^2-m\bar x^2\Big)=\sum_t x_ty_t-m\bar x\bar y.
$$

Since $\sum_t(x_t-\bar x)^2=\sum_t x_t^2-m\bar x^2$ and $\sum_t(x_t-\bar x)(y_t-\bar y)=\sum_t x_ty_t-m\bar x\bar y$,

$$
\hat\beta=\frac{\sum_{t=1}^m (x_t-\bar x)(y_t-\bar y)}{\sum_{t=1}^m (x_t-\bar x)^2},
\qquad
\hat\alpha=\bar y-\hat\beta\bar x,
$$

which is the stated formula. (The second-order conditions hold since $Q$ is a convex
quadratic in $(a,b)$, so this stationary point is the unique minimizer.)


In [ ]:
# Verification against statsmodels OLS
X = sm.add_constant(x)
ols_model = sm.OLS(y, X).fit()

comparison = pd.DataFrame({
    "closed_form": [alpha_hat, beta_hat],
    "statsmodels": [ols_model.params[0], ols_model.params[1]],
}, index=["alpha_hat", "beta_hat"])
comparison


In [ ]:
print(ols_model.summary())


### c) Interpretation and a fitted value

In [ ]:
x_new = 0.01
y_fit_new = alpha_hat + beta_hat * x_new

print(f"beta_hat  = {beta_hat:.4f}")
print(f"alpha_hat = {alpha_hat:.6f}")
print(f"Fitted AAPL log-return when SPY log-return = {x_new}: {y_fit_new:.6f}")


**Interpretation of $\hat\beta$.** $\hat\beta$ estimates AAPL's sensitivity to
market-wide (SPY) moves: on average, a 1 percentage-point change in the SPY log-return is
associated with a $\hat\beta$ percentage-point change in the AAPL log-return, holding the
idiosyncratic term $u_t$ at its conditional mean of zero. This is exactly the CAPM notion of
a stock's **market beta** — $\hat\beta>1$ indicates AAPL is more volatile than (amplifies)
the market, $\hat\beta<1$ indicates it is less volatile, and $\hat\alpha$ is the average
daily AAPL return unexplained by the market (an estimate of daily "alpha"). The fitted
value printed above is the model's predicted AAPL log-return on a day where SPY returns
1%.


### d) Decomposition of variance: TSS = ESS + RSS

In [ ]:
y_hat = alpha_hat + beta_hat * x

TSS = np.sum((y - y_bar) ** 2)
ESS = np.sum((y_hat - y_bar) ** 2)
RSS = np.sum((y - y_hat) ** 2)

R2 = ESS / TSS

print(f"TSS       = {TSS:.6f}")
print(f"ESS + RSS = {ESS + RSS:.6f}")
print(f"RSS       = {RSS:.6f}")
print(f"|TSS - (ESS+RSS)| = {abs(TSS - (ESS + RSS)):.2e}")
print()
print(f"R^2 (via ESS/TSS)   = {R2:.4f}")
print(f"R^2 (via 1-RSS/TSS) = {1 - RSS / TSS:.4f}")
print(f"R^2 (statsmodels)   = {ols_model.rsquared:.4f}")


The two computations of $R^2$ agree numerically (up to floating-point error) with
each other and with `statsmodels`' reported $R^2$, and $TSS=ESS+RSS$ holds because the OLS
residuals are orthogonal to the fitted values (an implication of the normal equations
derived in part (b)). $R^2$ reports the fraction of the variance in AAPL's daily returns
that is linearly explained by SPY's daily returns.
